### **Table of Content**
- Chapter 1: Introduction to Open AI
- Chapter 2: Summarizing & Editing Text
- Chapter 3: Text Generation
- Chapter 4: Shot Prompting (Give LLM Examples)
- Chapter 5: Chat Roles and System Messages
- Chapter 6: Utilizing The Assistant Role (Single-turn Tasks)
- Chapter 7: Multi-turn Conversations with GPT

#### **1.0 Introduction to Open AI** 

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI, RateLimitError  # Import the specific error & OpenAI

In [5]:
# 1. Setup path and load .env
# We use Path.cwd() to get the folder where the notebook is located
# Then go up two levels (parent.parent) to reach the root .env
env_path = Path.cwd().parent.parent / '.env'
load_dotenv(dotenv_path=env_path)

# 2. Initialize Client
# IMPORTANT: Ensure "OPENAI_API_KEY" is the name used inside your .env file
api_key = os.getenv("OPENAI_API_KEY") 
client = OpenAI(api_key=api_key)

try:
    # 3. Attempt the API call
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=100,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Hello!"}
        ]
    )
    print(response.choices[0].message.content)

except RateLimitError as e:
    if "insufficient_quota" in str(e):
        print("\n[!] ERROR: Insufficient Balance")
        print("Your OpenAI account has no credits left or your trial has expired.")
        print("Please add funds here: https://platform.openai.com/settings/organization/billing")
    else:
        print(f"\n[!] Rate limit reached: {e}")

except Exception as e:
    print(f"\n[!] An unexpected error occurred: {e}")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

#### **2.0 Summarizing & Editing Text**

In [ ]:
text = ""
prompt = f"""

: {text}
""" 

max_completion_tokens = 100  # Set a reasonable token limit for the response

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role":"user",
            "content":prompt
        }
    ],
    max_completion_tokens=max_completion_tokens

)

# Estimate token usage (input + output)
# Calculate cost based on token usage
input_token_price = 0.15 / 1_000_000  # Example price per input token
output_token_price = 0.20 / 1_000_000  # Example price per output token

# Extract token counts from the response (if available)
input_tokens = response.usage.input_tokens if hasattr(response, 'usage') else 0
output_tokens = response.usage.output_tokens if hasattr(response, 'usage') else 0
total_cost = input_token_price * input_tokens + output_token_price * output_tokens

print(f"Input Tokens: {input_tokens}, Output Tokens: {output_tokens}, Total Cost: ${total_cost:.6f}")

#### **3.0 Text Generation**

In [ ]:
prompt = f"""

: {text}
""" 

max_completion_tokens = 100  # Set a reasonable token limit for the response    

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role":"user",
            "content":prompt
        }
    ],
    temperature=0.7, # controls randomness of the output
    max_completion_tokens=max_completion_tokens
)

print(response.choices[0].message.content)

#### **4.0 Shot Prompting (Give LLM Examples)**

In [ ]:
prompt ="""
classify the sentinment of the statements ranging from 1-5 (1 poor-happy 5):
1. Unbelievably good!
2. Shoes fell apart on the second use.
3. The shoes look nice, but they aren't very comfortable.
4. Can't wait to show them off!
""" 

max_completion_tokens = 100  # Set a reasonable token limit for the response    

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role":"user",
            "content":prompt
        }
    ],
    temperature=0.7, # controls randomness of the output
    max_completion_tokens=max_completion_tokens
)

print(response.choices[0].message.content)

#### **5.0 Chat Roles and System Messages**

Suggested Google framework **PARTS** for good practice of prompt engineering:<br>
- Persona
- Aim
- Recipient
- Theme
- Structure

In [ ]:
# System: Context and instructions for the assistant, can define the assistant's behavior and constraints
# User: Instruct the assistant on what to do
# Assistant: Response to user instructions, can provide example


sys_msg = """
You are a Python Programmer tutor for beginners.
"""
prompt ="""
What is the difference between a list and a tuple in Python?
""" 

max_completion_tokens = 100  # Set a reasonable token limit for the response    

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role":"system", # PERSONA
            "content":sys_msg # add guardians to restrict the assistant's behavior
        },
        {
            "role":"user", # AIM
            "content":prompt
        }
    
    ],
    temperature=0.7, # controls randomness of the output
    max_completion_tokens=max_completion_tokens
)

print(response.choices[0].message.content)

#### **6.0 Utilizing The Assistant Role (Single-turn Tasks)**

In [ ]:
# Single-turn Tasks

sys_msg = """
You are a Python Programmer tutor for beginners.
"""
prompt ="""
What is the difference between a list and a tuple in Python?
""" 

max_completion_tokens = 100  # Set a reasonable token limit for the response    

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role":"system", # PERSONA
            "content":sys_msg # add guardians to restrict the assistant's behavior
        },

        {
            "role":"user", # AIM
            "content":"How do I create a virtual environment in Python?"
        },

        {
            "role":"assistant", 
            "content":"To create a virtual environment in Python, you can use the 'python -m venv' command followed by the name of your virtual environment."
        },

        {
            "role":"user", # AIM
            "content":prompt
        }
    
    ],
    temperature=0.7, # controls randomness of the output
    max_completion_tokens=max_completion_tokens
)

print(response.choices[0].message.content)

#### **7.0 Multi-turn Conversations with GPT**

In [ ]:
# multi-turn Tasks

# Initialize message history with system prompt
msg = [
    {
        "role":"system", # PERSONA
        "content":"You are a AI engineer tutor who provides short, simple explanations." # add guardians to restrict the assistant's behavior
    }
]

# List of user questions for multi-turn conversation
user_qs = [
    "What is a neural network?",
    "How does backpropagation work?",
    "What is the difference between supervised and unsupervised learning?"
    ]

# Iterate through user questions, append to the message history, and get responses from the assistant
for q in user_qs:
    # Display the user's question
    print(f"\nUser: {q}")
    user_dict = {
        "role":"user",
        "content":q
    }
    msg.append(user_dict)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=msg,
        temperature=0.7, # controls randomness of the output
        max_completion_tokens=100
    )

    assistant_dict = {
        "role":"assistant",
        "content":response.choices[0].message.content
    }
    msg.append(assistant_dict)
    # Print the assistant's response
    print(f"Assistant: {response.choices[0].message.content}")

